# Step 4: frozen-feature probe: which normalisation x crop x backbone carries pCR signal? (GPU, ~30 min)
Fine-tuning one ViT costs ~30 min and its val AUC is noisy (177 patients, 95% CI about +-0.08). So first,
every input variant goes through **frozen** backbones. Each patient's slice features are averaged and an L2
logistic regression is fitted. Variants are ranked by **5-fold CV AUC on the ~1,100 training patients** (low
variance), and val AUC is shown as a confirmation. The best rows tell step 5 what to fine-tune.

1. Session options: **GPU T4 x2** (backbones are spread over both GPUs), Internet **ON**, secret `GITHUB_TOKEN`
   (or attach the code Dataset).
2. **+ Add Input**: `mnj-cache2-ispy1`, `mnj-cache2-ispy2`, `mnj-cache2-duke` (all three = all labeled patients).
3. First run: `LIMIT = 60` (a few minutes) to check everything works, then `LIMIT = 0`, Save & Run All.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
BACKBONES = ["mae", "dinov2_b"]   # also: "vit_sup", "dino_b16", "dinov2_s"
NORMS = ["joint_pct", "joint_wide", "per_phase", "joint_tumor", "pre_ref", "zscore", "sub", "kinetic", "early3"]
CROPS = ["fixed", "tumor"]          # also: "mm"
K = 8                               # slices per patient (central 60% of the tumour)
PREFER = "v2"
LIMIT = 0                           # patients per split; 60 for a quick check
PUSH_RESULTS = False
NOTEBOOK_NAME = "probe"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob
    hits = [h for d in range(1, 6) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    assert hits, ("no GITHUB_TOKEN secret and no code Dataset attached. Either add the secret (Add-ons -> Secrets) "
                  "or attach the Dataset made from mnj-code-vaib.zip (see kaggle/vaib/README.md)")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import time
OUTD = f"/kaggle/working/probe_{time.strftime('%m%d-%H%M')}"
cmd = [sys.executable, "-m", "src.vitonly.probe", "--backbones", *BACKBONES, "--norms", *NORMS, "--crops", *CROPS,
       "--k", str(K), "--prefer", PREFER, "--out", OUTD, "--cache-root", "/kaggle/input", "/kaggle/working"]
if LIMIT: cmd += ["--limit", str(LIMIT)]
print(" ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f"{OUTD}/probe_results.csv")
cols = ["backbone", "norm", "crop", "pool", "cv_auc", "cv_auc_sd", "val_auc", "val_ci_lo", "val_ci_hi", "val_auc_within"]
cols += [c for c in df.columns if c.startswith("val_auc_") and c != "val_auc_within"]
pd.set_option("display.max_rows", 300, "display.width", 220)
display(df[cols].round(3).head(30))
ref = df[df["pool"].isin(["cohort_only", "kinetic_handcrafted"])]
display(ref[cols].round(3))
best = df[df.backbone != "-"].groupby(["norm", "crop"]).cv_auc.max().unstack()
best.plot.barh(figsize=(8, 5), xlim=(0.45, max(0.75, best.max().max() + 0.02)))
plt.axvline(float(ref[ref["pool"] == "cohort_only"].cv_auc.iloc[0]), color="grey", ls="--", label="cohort only")
plt.xlabel("best 5-fold CV AUC on train"); plt.legend(); plt.tight_layout(); plt.show()
top = df[df.backbone != "-"].iloc[0]
print(f"TOP: backbone={top.backbone} norm={top.norm} crop={top.crop} pool={top['pool']} "
      f"cv {top.cv_auc:.3f}, val {top.val_auc:.3f} -> use in step 5")
os.makedirs(f"/kaggle/working/results/{PERSON}/{os.path.basename(OUTD)}", exist_ok=True)
for f in ("probe_results.csv", "probe_val_preds.csv"):
    shutil.copy(f"{OUTD}/{f}", f"/kaggle/working/results/{PERSON}/{os.path.basename(OUTD)}/{f}")

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")